# Joint Distributions

So far every distribution described **one** variable — the height of a population, or its age. But what if you want to look at two variables *together* and see how they combine? For that you need a **joint distribution**: a probability for every possible *pair* of values, $P_{XY}(x, y) = P(X = x,\ Y = y)$. This notebook builds one from a small dataset of children's ages and heights, introduces the notation, and organizes everything into the joint table that makes such questions easy. It is based on the first video of Lesson 4 (Probability Distributions with Multiple Variables).

## 1. Introduction

A dataset rarely has just one column. Ages *and* heights, ad budgets *and* sales, processing time *and* latency — the interesting questions are usually about how two variables relate: are tall children the older ones? Do bigger budgets come with bigger sales? Single-variable distributions cannot answer this; you need the distribution of the *pair*.

After completing this notebook you will be able to:

- Build the single-variable PMFs of two features and see why they are not enough.
- Answer questions like "what is the probability a child is 9 years old **and** 49 inches tall?"
- Read and write the joint-distribution notation $P_{XY}(x, y) = P(X = x, Y = y)$.
- Organize data into a **joint counts table** and turn it into a **joint PMF**.

## 2. Theory

### Two variables, one dataset

Consider a simple dataset of 10 children, aged 7 to 10. Counting children per age:

| age | 7 | 8 | 9 | 10 |
|---|---|---|---|---|
| count | 3 | 2 | 4 | 1 |
| probability | 0.3 | 0.2 | 0.4 | 0.1 |

(dividing the counts by 10 turns them into probabilities — e.g. $P(\text{age} = 9) = 0.4$; this is the kids example from Lesson 3, back again).

For the *same* children we also record heights, rounded to the nearest inch:

| height (in) | 45 | 46 | 47 | 48 | 49 | 50 |
|---|---|---|---|---|---|---|
| count | 1 | 2 | 2 | 0 | 3 | 2 |
| probability | 0.1 | 0.2 | 0.2 | 0 | 0.3 | 0.2 |

(no child is 48 inches tall, but we keep the value on the axis to complete it; $P(\text{height} = 47) = 0.2$).

### A question the two PMFs cannot answer alone

**What is the probability that a child is 9 years old *and* 49 inches tall?**

From the age PMF: four children are 9 years old. Looking at those four children's heights: only **three** of them are 49 inches tall. Dividing by the 10 kids total:

$$
P(\text{age} = 9 \text{ and height} = 49) = \frac{3}{10}
$$

Note that neither single-variable PMF contains this number — we had to go back to the *individual children* to link the two features. That linking is exactly what the joint distribution stores.

### Notation

Call age $X$ and height $Y$. The joint probability is written

$$
P_{XY}(x, y) = P(X = x,\ Y = y)
$$

— capital letters name the variables, lowercase letters are the specific values you want them to take. Our example: $P_{XY}(9, 49) = 0.3$.

### The joint table

Calculations get much easier if we organize the data properly: a table with one row per age and one column per height, each cell counting the children with that exact combination:

| count | 45 | 46 | 47 | 48 | 49 | 50 |
|---|---|---|---|---|---|---|
| **7** | 1 | 2 | 0 | 0 | 0 | 0 |
| **8** | 0 | 0 | 2 | 0 | 0 | 0 |
| **9** | 0 | 0 | 0 | 0 | 3 | 1 |
| **10** | 0 | 0 | 0 | 0 | 0 | 1 |

Divide everything by 10 and the table of counts becomes the **joint probability mass function** — a probability for each possible combination of $x$ and $y$. All the possible combinations together are the **joint distribution**; since both age and height take separate, countable values, this one is a **discrete** joint distribution.

Like every PMF it satisfies: every entry $\ge 0$, and all entries **sum to 1**.

## 3. Worked Examples

### Example 1: reading the table

$$
P_{XY}(9, 49) = \frac{3}{10} = 0.3
$$

— three children sit in the (age 9, height 49) cell.

### Example 2: an empty cell

**What is the probability that a child is 8 years old and 48 inches tall?**

$$
P_{XY}(8, 48) = 0
$$

— there are zero children with that combination (the cell is 0). Perfectly valid: most cells of a joint table are often empty.

### Example 3: one more lookup

$$
P_{XY}(7, 46) = \frac{2}{10} = 0.2
$$

— two children are 7 years old and 46 inches tall.

### Example 4: the sanity check

Summing every cell: $\frac{1+2+2+3+1+1}{10} = \frac{10}{10} = 1$. Each child lands in exactly one cell, so the joint PMF distributes the total probability of 1 across the combinations.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

# clear any stray display settings from the session so float tables print in full
for _option in ("display.float_format", "display.precision", "display.chop_threshold"):
    pd.reset_option(_option)

rng = np.random.default_rng(42)

# The 10 children as raw data: one (age, height) pair each
children = pd.DataFrame({
    "age":    [7,  7,  7,  8,  8,  9,  9,  9,  9, 10],
    "height": [45, 46, 46, 47, 47, 49, 49, 49, 50, 50],
})
print(children.T)

### From raw data to the joint table

`pd.crosstab` builds the joint counts in one line; dividing by $n$ gives the joint PMF.

In [ ]:
ages = [7, 8, 9, 10]
heights = [45, 46, 47, 48, 49, 50]

joint_counts = pd.crosstab(children["age"], children["height"])
joint_counts = joint_counts.reindex(index=ages, columns=heights, fill_value=0)
print("joint counts:")
print(joint_counts)

joint_pmf = joint_counts / len(children)
print("\njoint PMF (divide by 10):")
print(joint_pmf)
print(f"\nsum of all cells: {joint_pmf.values.sum()}   (a valid PMF sums to 1)")

### Answering the video's questions from the table

In [ ]:
def P(age, height):
    return joint_pmf.loc[age, height]

print(f"P(age = 9,  height = 49) = {P(9, 49)}")
print(f"P(age = 8,  height = 48) = {P(8, 48)}   (empty cell -> probability 0)")
print(f"P(age = 7,  height = 46) = {P(7, 46)}")

### The single-variable PMFs live inside the joint table

Sum each row and you recover the age PMF; sum each column and you recover the height PMF. (These row/column sums have a name — *marginal distributions* — coming up in a later video.)

In [ ]:
age_pmf = joint_pmf.sum(axis=1)
height_pmf = joint_pmf.sum(axis=0)

print("row sums (the age PMF):")
print(age_pmf)
print("\ncolumn sums (the height PMF):")
print(height_pmf)

### A second example: two fair dice

For two *independent* fair dice, every one of the 36 (first, second) combinations is equally likely — the joint PMF is a flat table of $\tfrac{1}{36}$, very unlike the children's concentrated table.

In [ ]:
rng = np.random.default_rng(42)

rolls = pd.DataFrame(rng.integers(1, 7, size=(100_000, 2)), columns=["die1", "die2"])
dice_joint = pd.crosstab(rolls["die1"], rolls["die2"], normalize=True)
print("empirical joint PMF of two dice (theory: 1/36 = 0.0278 everywhere):")
print(dice_joint.round(4))

## 5. Visualizations

### The two single-variable histograms

The video's starting point: each variable on its own (note the empty 48-inch bar kept on the axis).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(age_pmf.index, age_pmf.values, color="tab:blue", edgecolor="black")
axes[0].set_title("PMF of age")
axes[0].set_xlabel("age (years)")
axes[0].set_xticks(ages)

axes[1].bar(height_pmf.index, height_pmf.values, color="tab:orange", edgecolor="black")
axes[1].set_title("PMF of height (note the empty 48)")
axes[1].set_xlabel("height (inches)")
axes[1].set_xticks(heights)

for ax in axes:
    ax.set_ylabel("probability")
    ax.grid(True, alpha=0.3, axis="y")
plt.suptitle("Each variable separately -- the joint question needs more")
plt.tight_layout()
plt.show()

### The joint PMF as a heatmap

The video's table as a picture: each cell colored by its probability, with the values annotated.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
im = ax.imshow(joint_pmf.values, cmap="Blues", vmin=0, vmax=0.35)

ax.set_xticks(range(len(heights)))
ax.set_xticklabels(heights)
ax.set_yticks(range(len(ages)))
ax.set_yticklabels(ages)
ax.set_xlabel("height y (inches)")
ax.set_ylabel("age x (years)")
for i in range(len(ages)):
    for j in range(len(heights)):
        v = joint_pmf.values[i, j]
        ax.text(j, i, f"{v:.1f}", ha="center", va="center",
                color="white" if v > 0.2 else "black")
ax.set_title("Joint PMF $P_{XY}(x, y)$ of age and height")
fig.colorbar(im, label="probability")
plt.tight_layout()
plt.show()

### The joint distribution in 3D

The same table as a bar landscape — the two-variable analogue of the histograms above.

In [ ]:
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")

xs, ys = np.meshgrid(range(len(heights)), range(len(ages)))
xs, ys = xs.ravel(), ys.ravel()
zs = joint_pmf.values.ravel()

nonzero = zs > 0                      # draw bars only where probability exists
ax.bar3d(xs[nonzero] - 0.4, ys[nonzero] - 0.4, np.zeros(nonzero.sum()),
         0.8, 0.8, zs[nonzero], color="tab:blue", alpha=0.8, shade=True)
ax.set_xticks(range(len(heights)))
ax.set_xticklabels(heights)
ax.set_yticks(range(len(ages)))
ax.set_yticklabels(ages)
ax.set_xlabel("height y (inches)")
ax.set_ylabel("age x (years)")
ax.set_zlabel("probability")
ax.set_title("The joint PMF as 3D bars")
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** From the joint table, read off $P_{XY}(9, 50)$, $P_{XY}(10, 50)$, and $P_{XY}(10, 45)$.

**Exercise 2.** Using the joint PMF, compute the probability that a child is *at most* 8 years old **and** *at most* 47 inches tall (sum the relevant cells).

### Intermediate

**Exercise 3.** Compute $P(\text{age} = 9) \cdot P(\text{height} = 49)$ from the single-variable PMFs and compare it with the joint probability $P_{XY}(9, 49) = 0.3$. What does the mismatch tell you about age and height in this dataset? (Hint: recall the product rule for independent events.)

**Exercise 4.** Build the joint PMF of two variables from a *single* die roll: $X$ = the number rolled, $Y$ = its parity (0 = even, 1 = odd). Write the 6×2 table. How many nonzero cells are there per row, and why?

### Advanced

**Exercise 5.** For two fair dice, let $X$ = first die and $S$ = the **sum** of both. Build the joint PMF table of $(X, S)$ by enumeration, verify it sums to 1, and read off $P_{X S}(6, 7)$. Which cells are structurally zero, and why?

**Exercise 6.** Simulation meets theory: sample 100,000 children *with replacement* from the 10-kid dataset, build the empirical joint PMF with `pd.crosstab(..., normalize=True)`, and compare it with the exact table (max absolute difference). Then run the *wrong* simulation — draw age and height independently from their own PMFs — and compare that table with both the exact joint and the product $P(x)P(y)$. Why does sampling whole children preserve the joint structure while drawing the two variables separately does not?

## 7. Solutions

In [ ]:
# Exercise 1: three lookups
print(f"Exercise 1: P(9, 50)  = {P(9, 50)}")
print(f"            P(10, 50) = {P(10, 50)}")
print(f"            P(10, 45) = {P(10, 45)}   (no such child)")

In [ ]:
# Exercise 2: a rectangular event
young_short = joint_pmf.loc[[7, 8], [45, 46, 47]]
print("Exercise 2: cells with age <= 8 and height <= 47:")
print(young_short)
print(f"P(age <= 8 and height <= 47) = {young_short.values.sum()}")
print("All five children of ages 7-8 happen to be at most 47 inches tall.")

In [ ]:
# Exercise 3: joint vs product of singles
product = age_pmf[9] * height_pmf[49]
print(f"Exercise 3: P(age=9) * P(height=49) = 0.4 * 0.3 = {product:.2f}")
print(f"Joint P(9, 49) = {P(9, 49)}")
print("0.3 > 0.12: the joint probability is much larger than the product, so age")
print("and height are NOT independent -- knowing a child is 9 makes 49 inches far")
print("more likely. Joint distributions capture exactly this dependence.")

In [ ]:
# Exercise 4: die value and its parity
faces = np.arange(1, 7)
table = pd.DataFrame(0.0, index=faces, columns=[0, 1])
for f in faces:
    table.loc[f, f % 2] = 1 / 6
table.index.name = "X (roll)"
table.columns.name = "Y (parity)"
print("Exercise 4:")
print(table)
print("One nonzero cell per row: the roll DETERMINES the parity, so each value of")
print("X is compatible with exactly one value of Y -- total dependence.")

In [ ]:
# Exercise 5: first die and the sum
first = np.arange(1, 7)
sums = np.arange(2, 13)
joint_xs = pd.DataFrame(0.0, index=first, columns=sums)
for d1 in first:
    for d2 in first:
        joint_xs.loc[d1, d1 + d2] += 1 / 36
joint_xs.index.name = "X (die 1)"
joint_xs.columns.name = "S (sum)"
print("Exercise 5:")
print(joint_xs.round(3))
print(f"\ntotal = {joint_xs.values.sum():.3f},  P(X=6, S=7) = {joint_xs.loc[6, 7]:.4f} (= 1/36)")
print("Cells with S < X + 1 or S > X + 6 are structurally zero: the second die")
print("can only add 1 to 6 to the first.")

In [ ]:
# Exercise 6: resampling children keeps the joint structure
rng = np.random.default_rng(0)

idx = rng.integers(0, len(children), size=100_000)
# reset_index: iloc keeps the original labels, and crosstab needs a unique index
resampled = children.iloc[idx].reset_index(drop=True)
empirical = pd.crosstab(resampled["age"], resampled["height"], normalize=True)
empirical = empirical.reindex(index=ages, columns=heights, fill_value=0.0)

print("Exercise 6: empirical joint PMF from 100,000 resampled children:")
print(empirical.round(4))
print("\nexact joint PMF:")
print(joint_pmf)

diff = np.abs(empirical.values - joint_pmf.values).max()
print(f"\nmax |empirical - exact| = {diff:.4f}")
print("Each draw picks a WHOLE child -- an (age, height) pair together -- so the")
print("pairing survives. Sampling age and height from their separate PMFs would")
print("scramble the link and produce the (wrong) product table instead.")

In [ ]:
# Exercise 6 (continued): the WRONG simulation -- age and height drawn separately
rng = np.random.default_rng(1)

sep_age = rng.choice(ages, size=100_000, p=age_pmf.values)
sep_height = rng.choice(heights, size=100_000, p=height_pmf.values)

separate = pd.crosstab(pd.Series(sep_age, name="age"),
                       pd.Series(sep_height, name="height"), normalize=True)
separate = separate.reindex(index=ages, columns=heights, fill_value=0.0)
print("joint PMF estimated from SEPARATE, independent draws of age and height:")
print(separate.round(4))

product = pd.DataFrame(np.outer(age_pmf.values, height_pmf.values),
                       index=ages, columns=heights)
product.index.name, product.columns.name = "age", "height"
print("\nthe product table P(x) * P(y) that those draws converge to:")
print(product.round(3))

print(f"\nmax |separate - exact joint| = {np.abs(separate.values - joint_pmf.values).max():.4f}")
print(f"max |separate - product|     = {np.abs(separate.values - product.values).max():.4f}")
print("\nSeparate draws reproduce the PRODUCT of the marginals, not the joint: cell")
print("(9, 49) lands near 0.12 instead of 0.3, and impossible children appear --")
print("(7, 49) gets about 0.09 although no 7-year-old is 49 inches tall. More")
print("samples do not help; this simulation converges to the wrong table.")

print("\nrow sums of the separate-draw table vs the exact age PMF:")
print(pd.DataFrame({"separate": separate.sum(axis=1), "exact": age_pmf}).round(4))
print("\nBoth simulations get the MARGINALS right; only the paired one gets the")
print("interior right -- exactly the information a joint distribution adds.")


## 8. Conclusion

- One-variable distributions cannot answer questions about *combinations*; for that you need the **joint distribution** of the pair.
- Notation: $P_{XY}(x, y) = P(X = x, Y = y)$ — the probability that $X$ takes value $x$ **and** $Y$ takes value $y$ ($P_{XY}(9, 49) = 0.3$ for the children).
- The practical tool is the **joint table**: rows for $x$, columns for $y$, each cell a count; divide by $n$ for the **joint PMF**. Cells can be 0 (no 8-year-old is 48 inches); all cells sum to 1.
- Because both variables here take separate countable values, this is a **discrete** joint distribution.
- Sampling shows the same thing: resampling **whole children** reproduces the joint table, while drawing age and height from their own PMFs converges to the product table instead — right marginals, wrong interior.
- The single-variable PMFs are recoverable as row and column sums of the joint table — and the joint holds strictly *more* information: it records the dependence between the variables (here $P_{XY}(9,49) = 0.3 \ne 0.12 = P(9)P(49)$).

Next: continuous joint distributions, and then those row/column sums get their proper name — **marginal distributions**.